# ENSF 617 – Assignment 02  
## Garbage Classification Using Images and Text

**Group:** Group 8  
**Course:** ENSF 617 – Machine Learning  
**Assignment:** Garbage Classification Model - Programming Assignment

This notebook implements a multimodal garbage classification system using
image data and textual metadata. The model is trained and evaluated using
PyTorch, following a proper train/validation/test experimental setup.


## 1. Imports & Environment Setup

In [1]:
# Core
import os
import random
import numpy as np

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

# Vision
from torchvision import transforms

# Metrics & visualization
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys

# Point Colab exactly to the folder shown in your screenshot
project_path = "/content/drive/MyDrive/Garbage_Classification"
os.chdir(project_path)

# Add it to the system path just to be completely safe
sys.path.append(project_path)

print(f"Current working directory is now: {os.getcwd()}")

Current working directory is now: /content/drive/MyDrive/Garbage_Classification


In [4]:
!tar -xzf /content/drive/MyDrive/Garbage_Classification/garbage_data.tar.gz

## 2. Dataset Overview

The dataset consists of garbage images categorized into the following classes:

- Black
- Blue
- Green
- Other

The dataset is organized into pre-defined train, validation, and test splits.
Each sample contains:
- An image
- Associated textual information (metadata/description)


In [9]:
# dataset root, define and print classes

DATASET_ROOT = "/content/drive/MyDrive/Garbage_Classification/garbage_data"

# DATASET_ROOT = os.getenv("DATASET_ROOT", "/default/path")

#DATASET_ROOT = "/work/TALC/ensf617_2026w/garbage_data"

## 3. Data Preprocessing

We apply standard preprocessing steps to the image data, including resizing,
normalization, and optional data augmentation. Text data is tokenized or
vectorized before being passed to the model.


In [10]:
from src.preprocessing import get_image_transforms
image_transforms = get_image_transforms()

## 4. Dataset & DataLoader
We define a custom PyTorch Dataset to load both image and textual data,
and create DataLoaders for training, validation, and testing.


In [11]:
from src.dataset import GarbageDataset

train_dataset = GarbageDataset(split="train", transform=image_transforms["train"], dataset_root=DATASET_ROOT)
val_dataset   = GarbageDataset(split="val",   transform=image_transforms["val"],   dataset_root=DATASET_ROOT)
test_dataset  = GarbageDataset(split="test",  transform=image_transforms["test"],  dataset_root=DATASET_ROOT)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

TRAIN split loaded with 9527 samples.
VAL split loaded with 1492 samples.
TEST split loaded with 2580 samples.


## 5. Model Architecture
The model consists of:
- A CNN-based image encoder
- A text encoder for textual metadata
- A fusion layer combining both modalities
- A fully connected classifier for final prediction

We use transfer learning by initializing a ResNet-18 model pre-trained on ImageNet. The final classification layer is replaced to match our garbage dataset classes. We first freeze the feature extractor and train the classifier head.


In [14]:
from src.model import GarbageClassifier

classes = ["Black", "Blue", "Green", "Other"]

model = GarbageClassifier(num_classes=len(classes), text_feature_dim=len(classes))
model = model.to(device)
model

GarbageClassifier(
  (image_encoder): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine

## 6. Training Setup

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4) # using L2 regularization
num_epochs = 10

## 7. Training & Validation

In [ ]:
from src.train import train_model

history_stage1 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=num_epochs
)

Epoch [1/10] Train:  48%|████▊     | 142/298 [11:01<11:37,  4.47s/it, loss=0.399]

In [ ]:
# -------- Fine-Tuning Stage - transfer learning --------
print("Starting fine-tuning...")

# Unfreeze backbone
for param in model.image_encoder.parameters():
    param.requires_grad = True

# Smaller learning rate for fine-tuning
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)

history_stage2 = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=5
)

## 8. Training Curves

In [ ]:
#combining stage 1 and 2 histories
fine_tune_start = len(history_stage1["train_loss"])

history = {
    "train_loss": history_stage1["train_loss"] + history_stage2["train_loss"],
    "val_loss": history_stage1["val_loss"] + history_stage2["val_loss"]
}

#plt loss curve

plt.figure(figsize=(10, 4))

plt.plot(history["train_loss"], label="Train Loss")
plt.axvline(x=fine_tune_start-1, linestyle="--", label="Fine-tuning start") #line to visually show stage 1 and 2 transition
plt.plot(history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss (Transfer Learning)")
plt.legend()
plt.grid(True)

plt.show()

To mitigate overfitting, we employ:
- Data augmentation (random horizontal flip)
- Dropout in classifier
- L2 regularization (weight decay)
- Validation monitoring

## 9. Test Evaluation

In [ ]:
from src.evaluate import evaluate_model, plot_multiclass_roc_curve

y_true, y_pred, y_probs = evaluate_model(model, test_loader, device)

In [ ]:
print(classification_report(y_true, y_pred, target_names=classes))

In [ ]:
auc_score = roc_auc_score(y_true, y_probs, multi_class='ovr')
print(f"Multi-Class AUC-ROC Score (OVR): {auc_score:.4f}")

In [ ]:
# Plot the AUC-ROC curves
plot_multiclass_roc_curve(y_true, y_probs, classes)

## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=classes,
            yticklabels=classes,
            cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

## 11. Incorrect Classification Analysis
Below we visualize examples where the model made incorrect predictions,
including the true label and predicted label.

In [ ]:
from src.evaluate import show_incorrect_predictions

show_incorrect_predictions(
    model=model,
    dataset=test_dataset,
    classes=classes,
    device=device,
    num_examples=8
)

## 12. Conclusion
In this assignment, we implemented a multimodal garbage classification system
using both image and textual information. The model was trained and evaluated
using a proper experimental setup with train, validation, and test splits.

Future improvements could include deeper CNN architectures, improved text
embeddings, and additional data augmentation.
